In [5]:
import time
import simtk.openmm as mm
import simtk.openmm.app as app
import simtk.unit as unit
import parmed as pmd
from simtk.openmm import Platform, VerletIntegrator


class PerResidueEnergyDecomposition:
    def __init__(self, simulation, system, by_pass_bonds_index, by_pass_atoms):
        self.forces = {system.getForce(index).__class__.__name__: system.getForce(index) for index in range(system.getNumForces())}
        self.by_pass_bonds_index = by_pass_bonds_index
        self.simulation = simulation
        self.by_pass_atoms = by_pass_atoms

    def NonBondedForce_Decomposition(self):
        # Find the index of the NonbondedForce in the system
        nonbonded_force_index = -1
        for index in range(self.simulation.system.getNumForces()):
            if isinstance(self.simulation.system.getForce(index), mm.NonbondedForce):
                nonbonded_force_index = index
                break

        if nonbonded_force_index == -1:
            raise ValueError("No NonbondedForce found in the system.")

        # Create a new NonbondedForce
        new_nonbonded_force = mm.NonbondedForce()

        # Retrieve parameters and add to the new force
        for index in range(self.forces['NonbondedForce'].getNumParticles()):
            charge, sigma, epsilon = self.forces['NonbondedForce'].getParticleParameters(index)
            exclude = (index in [atom.index for atom in self.by_pass_atoms])
            new_nonbonded_force.addParticle(0 * charge if exclude else charge, sigma, 0 * epsilon if exclude else epsilon)

        # Copy over exceptions
        for index in range(self.forces['NonbondedForce'].getNumExceptions()):
            particle1, particle2, chargeProd, sigma, epsilon = self.forces['NonbondedForce'].getExceptionParameters(index)
            if particle1 not in [atom.index for atom in self.by_pass_atoms] and particle2 not in [atom.index for atom in self.by_pass_atoms]:
                new_nonbonded_force.addException(particle1, particle2, chargeProd, sigma, epsilon)

        # Remove the old nonbonded force
        self.simulation.system.removeForce(nonbonded_force_index)

        # Add the new NonbondedForce
        self.simulation.system.addForce(new_nonbonded_force)

        # Reinitialize the context to apply changes
        self.simulation.context.reinitialize()

        # Optional: Update parameters in context (not needed if reinitializing)
        # new_nonbonded_force.updateParametersInContext(self.simulation.context)




    def HarmonicBondForce_Decomposition(self):
        harmonic_bond_force = self.forces['HarmonicBondForce']
        for i in range(harmonic_bond_force.getNumBonds()):
            p1, p2, length, k = harmonic_bond_force.getBondParameters(i)
            exclude = (p1 in self.by_pass_bonds_index or p2 in self.by_pass_bonds_index)
            harmonic_bond_force.setBondParameters(i, p1, p2, length, 0 if exclude else k)
        harmonic_bond_force.updateParametersInContext(self.simulation.context)

    def HarmonicAngleForce_Decomposition(self):
        harmonic_angle_force = self.forces['HarmonicAngleForce']
        for i in range(harmonic_angle_force.getNumAngles()):
            p1, p2, p3, angle, k = harmonic_angle_force.getAngleParameters(i)
            exclude = (p1 in self.by_pass_bonds_index or p2 in self.by_pass_bonds_index or p3 in self.by_pass_bonds_index)
            harmonic_angle_force.setAngleParameters(i, p1, p2, p3, angle, 0 if exclude else k)
        harmonic_angle_force.updateParametersInContext(self.simulation.context)

    def PeriodicTorsionForce_Decomposition(self):
        periodic_torsion_force = self.forces['PeriodicTorsionForce']
        for i in range(periodic_torsion_force.getNumTorsions()):
            p1, p2, p3, p4, periodicity, phase, k = periodic_torsion_force.getTorsionParameters(i)
            exclude = (p1 in self.by_pass_bonds_index or p2 in self.by_pass_bonds_index or p3 in self.by_pass_bonds_index or p4 in self.by_pass_bonds_index)
            periodic_torsion_force.setTorsionParameters(i, p1, p2, p3, p4, periodicity, phase, 0 if exclude else k)
        periodic_torsion_force.updateParametersInContext(self.simulation.context)


def get_residue_atoms(index, modeller):
    residues = list(modeller.topology.residues())
    indicated_res_atoms = residues[index].atoms()
    RESIDUE_NAME = residues[index].name
    return list(indicated_res_atoms), RESIDUE_NAME


def run_mmgbsa(pdb_file, res_indices_list):
    start_time = time.time()
    print('Loading PDB file...')

    # Load the PDB file and create a modeller
    pdb = app.PDBFile(pdb_file)
    modeller = app.Modeller(pdb.topology, pdb.positions)

    for res_num in res_indices_list:
        calculate_atoms, selected_res_name = get_residue_atoms(res_num, modeller)
        by_pass_atoms = [atom for atom in modeller.topology.atoms() if atom not in calculate_atoms]

        # Identify bonds to bypass
        bonds = list(modeller.topology.bonds())
        by_pass_bonds_index = [bond[0].index for atom in calculate_atoms for bond in bonds if atom.residue != bond[0].residue]

        # Create the system using the force field
        forcefield = app.ForceField('amber96.xml', 'spce.xml')
        system = forcefield.createSystem(modeller.topology, 
                                           nonbondedMethod=app.NoCutoff,
                                           constraints=app.HBonds, 
                                           rigidWater=True)

        # Setup CUDA platform for simulation
        platform = Platform.getPlatformByName('CUDA')
        properties = {'CudaPrecision': 'mixed'}

        # Setup integrator and simulation
        integrator = VerletIntegrator(2.0 * unit.femtoseconds)
        simulation = app.Simulation(modeller.topology, system, integrator, platform, properties)
        simulation.context.setPositions(modeller.positions)

        # Perform energy decomposition
        decomposition = PerResidueEnergyDecomposition(simulation, system, by_pass_bonds_index, by_pass_atoms)
        #decomposition.HarmonicBondForce_Decomposition()
        #decomposition.HarmonicAngleForce_Decomposition()
        decomposition.NonBondedForce_Decomposition()
        #decomposition.PeriodicTorsionForce_Decomposition()

        # Calculate and print energy contributions
        structure = pmd.load_file(pdb_file)
        all_energies = pmd.openmm.energy_decomposition_system(structure, system, nrg=unit.kilojoules_per_mole)

        print("\n###################################################\n")
        print("RES_NUM: %s" % res_num)
        total_energy = sum(energy[1] for energy in all_energies)
        for energy in all_energies:
            print(energy[0], energy[1])
        print('Total energy of %s Residue: %6.6f kilojoules_per_mole' % (selected_res_name, total_energy))
        print("\n--- %s seconds ---" % (time.time() - start_time))
        print("###################################################")


if __name__ == '__main__':
    start_time = time.time()
    run_mmgbsa('1aki_solv_eq.pdb', [0, 25])  # Adjust the filename and residue indices as needed
    end_time = time.time()
    print("Total time: %s seconds" % (end_time - start_time))


Loading PDB file...

###################################################

RES_NUM: 0
HarmonicBondForce 1614.408184736967
PeriodicTorsionForce 3563.1654052734375
CMMotionRemover 0.0
HarmonicAngleForce 3776.8577880859375
NonbondedForce 399.101318359375
Total energy of LYS Residue: 9353.532696 kilojoules_per_mole

--- 60.84678864479065 seconds ---
###################################################

###################################################

RES_NUM: 25
HarmonicBondForce 1614.408184736967
PeriodicTorsionForce 3563.1654052734375
CMMotionRemover 0.0
HarmonicAngleForce 3776.8577880859375
NonbondedForce 69.53382873535156
Total energy of GLY Residue: 9023.965207 kilojoules_per_mole

--- 127.85327672958374 seconds ---
###################################################
Total time: 127.90238904953003 seconds


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
data = np.loadtxt("md_log.txt", delimiter=',')

step = data[:,0]
potential_energy = data[:,1]
temperature = data[:,2]
volume = data[:,3]


plt.plot(step, potential_energy)
plt.xlabel("Step")
plt.ylabel("Potential energy (kJ/mol)")
plt.show()
plt.plot(step, temperature)
plt.xlabel("Step")
plt.ylabel("Temperature (K)")
plt.show()
plt.plot(step, volume)
plt.xlabel("Step")
plt.ylabel("Volume (nm^3)")
plt.show()